# Penske Logistics Analytics - Learning Guide

## Sections
1. Setup & Environment
2. Data Generation
3. Data Exploration
4. Performance Analysis
5. Resource Prediction (ML)
6. Customer Acquisition (ML)
7. GenAI Insights
8. Data Guardrails (LLM Protection)
9. RAG & Embeddings (Vector Search)
10. Using This Template with Real Data

---
## Section 1: Setup & Environment

In [ ]:
# Step 1: Setup Python path
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print("Python path configured!")

In [ ]:
# Step 2: Import libraries and configure matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 5]

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("Libraries imported!")
print("Matplotlib configured for inline display")

---
## Section 2: Data Generation

Generate synthetic logistics data.

In [ ]:
# Import data generator functions
from src.data_generator import (
    generate_fleet_operations,
    generate_warehouse_metrics,
    generate_customer_data,
    generate_delivery_performance,
    generate_regional_demand,
    generate_leads_data,
    REGIONS, SERVICE_TYPES
)

print("Data generator imported!")
print(f"Regions: {REGIONS}")
print(f"Service Types: {SERVICE_TYPES}")

In [ ]:
# Generate Fleet Operations Data
fleet_df = generate_fleet_operations(num_records=500)

print("FLEET OPERATIONS")
print(f"Shape: {fleet_df.shape}")
print(f"Columns: {list(fleet_df.columns)}")
fleet_df.head()

In [ ]:
# Generate Regional Demand Data (for forecasting)
demand_df = generate_regional_demand(num_records=500)

print("REGIONAL DEMAND")
print(f"Shape: {demand_df.shape}")
print(f"Columns: {list(demand_df.columns)}")
demand_df.head()

In [ ]:
# Generate Warehouse Metrics
warehouse_df = generate_warehouse_metrics(num_records=500)

print("WAREHOUSE METRICS")
print(f"Shape: {warehouse_df.shape}")
print(f"Columns: {list(warehouse_df.columns)}")
warehouse_df.head()

In [ ]:
# Generate Customer Data
customer_df = generate_customer_data(num_customers=200)

print("CUSTOMER DATA")
print(f"Shape: {customer_df.shape}")
print(f"Columns: {list(customer_df.columns)}")
customer_df.head()

In [ ]:
# Generate Leads Data
leads_df = generate_leads_data(num_leads=150)

print("LEADS DATA")
print(f"Shape: {leads_df.shape}")
print(f"Columns: {list(leads_df.columns)}")
leads_df.head()

In [ ]:
# Generate Delivery Performance
delivery_df = generate_delivery_performance(num_records=500)

print("DELIVERY PERFORMANCE")
print(f"Shape: {delivery_df.shape}")
print(f"Columns: {list(delivery_df.columns)}")
delivery_df.head()

---
## Section 3: Data Exploration

Explore and visualize the data.

In [ ]:
# Fleet Statistics
print("FLEET OPERATIONS - Statistics")
print("=" * 50)
fleet_df.describe()

In [ ]:
# TEST: Verify matplotlib works
import matplotlib.pyplot as plt
%matplotlib inline

x = [1, 2, 3, 4, 5]
y = [1, 4, 9, 16, 25]

plt.figure(figsize=(8, 4))
plt.bar(x, y, color='blue')
plt.title('TEST PLOT - If you see this, matplotlib works!')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()

In [ ]:
# Visualization: On-Time Rate by Region
plt.figure(figsize=(10, 5))
fleet_df.groupby('region')['on_time_rate'].mean().sort_values().plot(kind='barh', color='steelblue')
plt.xlabel('Average On-Time Rate')
plt.title('On-Time Delivery Rate by Region')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Shipment Volume by Region
plt.figure(figsize=(10, 5))
demand_df.groupby('region')['shipment_volume'].sum().sort_values().plot(kind='barh', color='green')
plt.xlabel('Total Shipment Volume')
plt.title('Shipment Volume by Region')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Customer Satisfaction Distribution
plt.figure(figsize=(10, 5))
customer_df['satisfaction_score'].hist(bins=20, color='orange', edgecolor='black')
plt.xlabel('Satisfaction Score')
plt.ylabel('Count')
plt.title('Customer Satisfaction Score Distribution')
plt.tight_layout()
plt.show()

---
## Section 4: Performance Analysis

Calculate KPIs and analyze performance.

In [ ]:
# Import Performance Analyzer and create datasets dictionary
from src.service_performance import ServicePerformanceAnalyzer

# Create datasets dictionary from generated data
datasets = {
    'fleet_operations': fleet_df,
    'delivery_performance': delivery_df,
    'warehouse_metrics': warehouse_df,
    'customer_data': customer_df,
    'regional_demand': demand_df
}

analyzer = ServicePerformanceAnalyzer(datasets)
print("Performance Analyzer initialized!")

In [ ]:
# Calculate KPIs from fleet data
print("KEY PERFORMANCE INDICATORS (KPIs)")
print("=" * 50)

# On-Time Delivery Rate
on_time_rate = fleet_df['on_time_rate'].mean() * 100
print(f"On-Time Delivery Rate: {on_time_rate:.1f}%")

# Fleet Utilization
fleet_utilization = fleet_df['load_capacity_used'].mean()
print(f"Fleet Utilization: {fleet_utilization:.1f}%")

# Average Fuel Efficiency
fuel_efficiency = fleet_df['miles_driven'].sum() / fleet_df['fuel_consumed'].sum()
print(f"Fuel Efficiency: {fuel_efficiency:.1f} MPG")

# Customer Satisfaction
avg_satisfaction = customer_df['satisfaction_score'].mean()
print(f"Customer Satisfaction: {avg_satisfaction:.1f}/10")

In [ ]:
# Regional Performance Summary
print("REGIONAL PERFORMANCE")
print("=" * 50)

regional_stats = fleet_df.groupby('region').agg({
    'on_time_rate': 'mean',
    'miles_driven': 'sum',
    'load_capacity_used': 'mean'
}).round(2)

regional_stats.columns = ['Avg On-Time Rate', 'Total Miles', 'Avg Utilization']
regional_stats

---
## Section 5: Resource Prediction (ML)

Use machine learning to forecast demand.

In [ ]:
# Import Resource Predictor (DemandForecaster)
from src.resource_prediction import DemandForecaster

predictor = DemandForecaster()
print("Demand Forecaster initialized!")

In [ ]:
# Prepare features
demand_df['date'] = pd.to_datetime(demand_df['date'])
prepared_data = predictor.prepare_features(demand_df)

print("Feature Engineering Results:")
print(f"Original columns: {len(demand_df.columns)}")
print(f"After engineering: {len(prepared_data.columns)}")

new_cols = set(prepared_data.columns) - set(demand_df.columns)
print(f"\nNew features: {sorted(new_cols)[:10]}...")  # Show first 10

In [ ]:
# Train Demand Forecasting Model
print("Training demand forecasting model...")

results = predictor.train_demand_model(
    demand_df, 
    target_col='shipment_volume',
    model_type='gradient_boosting'
)

print("\nMODEL RESULTS")
print("=" * 50)
print(f"MAE: {results['metrics']['mae']:.2f}")
print(f"RMSE: {results['metrics']['rmse']:.2f}")
print(f"MAPE: {results['metrics']['mape']:.2f}%")

In [ ]:
# Feature Importance
if 'shipment_volume' in predictor.feature_importance:
    importance = predictor.feature_importance['shipment_volume']
    
    print("TOP FEATURES:")
    for feature, score in list(importance.items())[:10]:
        print(f"  {feature}: {score:.4f}")
    
    # Plot
    plt.figure(figsize=(10, 6))
    features = list(importance.keys())[:10]
    scores = list(importance.values())[:10]
    plt.barh(features, scores, color='steelblue')
    plt.xlabel('Importance')
    plt.title('Top 10 Features for Demand Prediction')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance will be available after training the model.")

---
## Section 6: Customer Acquisition (ML)

Lead scoring and churn prediction.

In [ ]:
# Import Customer Acquisition modules
from src.customer_acquisition import LeadScorer, ChurnPredictor

print("Customer Acquisition modules imported!")

In [ ]:
# Lead Scoring
lead_scorer = LeadScorer()

print("Training Lead Scoring Model...")
lead_results = lead_scorer.train(leads_df)

print("\nLEAD SCORING RESULTS")
print("=" * 50)
print(f"Accuracy: {lead_results['metrics']['accuracy']:.2%}")
print(f"Precision: {lead_results['metrics']['precision']:.2%}")
print(f"Recall: {lead_results['metrics']['recall']:.2%}")
print(f"AUC-ROC: {lead_results['metrics']['auc_roc']:.2%}")

In [ ]:
# Score leads
scored_leads = lead_scorer.score_leads(leads_df.head(20))

print("TOP SCORED LEADS:")
scored_leads.sort_values('lead_score', ascending=False).head(10)

In [ ]:
# Churn Prediction
churn_predictor = ChurnPredictor()

print("Training Churn Prediction Model...")
churn_results = churn_predictor.train(customer_df)

print("\nCHURN PREDICTION RESULTS")
print("=" * 50)
print(f"Accuracy: {churn_results['metrics']['accuracy']:.2%}")
print(f"Precision: {churn_results['metrics']['precision']:.2%}")
print(f"Recall: {churn_results['metrics']['recall']:.2%}")
print(f"AUC-ROC: {churn_results['metrics']['auc_roc']:.2%}")

In [ ]:
# Predict churn risk
churn_risk = churn_predictor.predict_churn_risk(customer_df.head(20))

print("CHURN RISK ASSESSMENT:")
churn_risk[['customer_id', 'company_name', 'churn_probability', 'risk_level']].head(10)

---
## Section 7: GenAI Insights

Generate insights using LLM (mock mode without API key).

In [ ]:
# Configure OpenAI API Key
import os

# Set your API key here (replace the placeholder with your actual key)
# WARNING: Do not commit real API keys to version control!
OPENAI_KEY = "your-openai-api-key-here"

os.environ['OPENAI_API_KEY'] = OPENAI_KEY

# Verify key is set
if OPENAI_KEY.startswith("sk-"):
    print("OpenAI API Key configured!")
else:
    print("WARNING: OpenAI API Key not set - will use mock responses")
    print("Replace OPENAI_KEY with your actual API key (starts with 'sk-')")

In [ ]:
# Import GenAI module (uses the API key from environment)
from src.genai_insights import InsightGenerator

insight_gen = InsightGenerator()

if insight_gen.client:
    print("Mode: OpenAI API (GPT-4) - Real AI responses enabled!")
else:
    print("Mode: Mock responses (no API key or openai package not installed)")

In [ ]:
# 5 Sample Questions for AI Agent Chat
# Create KPIs context for questions
kpis = {
    'on_time_rate': on_time_rate,
    'fleet_utilization': fleet_utilization,
    'customer_satisfaction': avg_satisfaction,
    'fuel_efficiency': fuel_efficiency
}

# Define 5 sample questions
questions = [
    "What are the top 3 areas that need improvement based on our KPIs?",
    "How can we improve our on-time delivery rate?",
    "What strategies should we implement to increase fleet utilization?",
    "Analyze our customer satisfaction score and suggest improvements.",
    "What cost optimization opportunities exist in our logistics operations?"
]

print("=" * 60)
print("AI AGENT CHAT - 5 SAMPLE QUESTIONS")
print("=" * 60)

for i, question in enumerate(questions, 1):
    print(f"\n{'='*60}")
    print(f"QUESTION {i}: {question}")
    print("-" * 60)
    answer = insight_gen.answer_question(question, kpis)
    print(f"ANSWER:\n{answer}")

---
## Section 8: Data Guardrails (LLM Protection)

Protect sensitive/real data from being exposed to external LLMs.

**Key Principles:**
- Never send PII (Personally Identifiable Information) to LLMs
- Anonymize or mask sensitive fields before LLM processing
- Use aggregated statistics instead of raw data
- Implement data classification and filtering

In [ ]:
# Data Guardrails Class - Protect sensitive data from LLM exposure
import re
import hashlib
from typing import Dict, List, Any

class DataGuardrail:
    """
    Protects sensitive data from being sent to external LLMs.
    Implements PII detection, masking, and data sanitization.
    """
    
    # Define sensitive field patterns
    SENSITIVE_FIELDS = [
        'customer_id', 'company_name', 'email', 'phone', 'address',
        'ssn', 'tax_id', 'credit_card', 'bank_account', 'driver_id',
        'employee_id', 'contract_value', 'salary', 'password'
    ]
    
    # PII regex patterns
    PII_PATTERNS = {
        'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        'phone': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
        'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
        'credit_card': r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b'
    }
    
    def __init__(self):
        self.masked_count = 0
        self.detected_pii = []
        
    def mask_value(self, value: str, field_name: str = '') -> str:
        """Mask a sensitive value"""
        if pd.isna(value):
            return value
        str_val = str(value)
        # Create a hash-based anonymous ID
        masked = hashlib.md5(str_val.encode()).hexdigest()[:8]
        return f"[MASKED-{masked}]"
    
    def detect_pii_in_text(self, text: str) -> Dict[str, List[str]]:
        """Detect PII patterns in text"""
        found_pii = {}
        for pii_type, pattern in self.PII_PATTERNS.items():
            matches = re.findall(pattern, str(text))
            if matches:
                found_pii[pii_type] = matches
        return found_pii
    
    def sanitize_dataframe(self, df: pd.DataFrame, mask_fields: List[str] = None) -> pd.DataFrame:
        """
        Sanitize a DataFrame by masking sensitive fields.
        Returns a copy - original data is NOT modified.
        """
        # Create a copy to avoid modifying original
        safe_df = df.copy()
        
        # Fields to mask
        fields_to_mask = mask_fields or self.SENSITIVE_FIELDS
        
        for col in safe_df.columns:
            col_lower = col.lower()
            # Check if column name matches sensitive patterns
            if any(sensitive in col_lower for sensitive in fields_to_mask):
                safe_df[col] = safe_df[col].apply(lambda x: self.mask_value(x, col))
                self.masked_count += len(safe_df)
                print(f"  [MASKED] Column: {col}")
        
        return safe_df
    
    def prepare_for_llm(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Prepare data dictionary for LLM - removes/masks sensitive info.
        Only sends aggregated statistics, not raw data.
        """
        safe_data = {}
        
        for key, value in data.items():
            key_lower = key.lower()
            
            # Skip sensitive fields entirely
            if any(sensitive in key_lower for sensitive in self.SENSITIVE_FIELDS):
                safe_data[key] = "[REDACTED]"
                continue
            
            # For DataFrames, only send summary statistics
            if isinstance(value, pd.DataFrame):
                safe_data[key] = {
                    'row_count': len(value),
                    'columns': list(value.columns),
                    'summary': 'DataFrame - raw data not sent to LLM'
                }
            # For numeric values, round to reduce precision
            elif isinstance(value, float):
                safe_data[key] = round(value, 2)
            else:
                safe_data[key] = value
                
        return safe_data

print("DataGuardrail class defined!")
print("This class protects sensitive data from LLM exposure.")

In [ ]:
# Example: Sanitize Customer Data before LLM processing
guardrail = DataGuardrail()

print("ORIGINAL DATA (contains sensitive info):")
print("-" * 50)
print(customer_df[['customer_id', 'company_name', 'contract_value', 'satisfaction_score']].head(3))

print("\n\nSANITIZED DATA (safe for LLM):")
print("-" * 50)
safe_customer_df = guardrail.sanitize_dataframe(customer_df)
print(safe_customer_df[['customer_id', 'company_name', 'contract_value', 'satisfaction_score']].head(3))

In [ ]:
# Example: Prepare KPIs for LLM (only aggregated stats, no raw data)
print("PREPARING DATA FOR LLM:")
print("=" * 50)

# Raw data that should NOT go to LLM
raw_data = {
    'customer_id': 'CUST-10001',
    'company_name': 'Acme Corporation',
    'contract_value': 1250000.50,
    'on_time_rate': 94.5,
    'fleet_utilization': 78.3,
    'customer_dataframe': customer_df
}

# Apply guardrail
safe_data = guardrail.prepare_for_llm(raw_data)

print("\nOriginal data keys:", list(raw_data.keys()))
print("\nSafe data for LLM:")
for key, value in safe_data.items():
    print(f"  {key}: {value}")

In [ ]:
# Safe LLM Query Function - Always use this instead of sending raw data
def safe_llm_query(insight_generator, question: str, data: Dict[str, Any]) -> str:
    """
    Wrapper function that applies guardrails before sending to LLM.
    ALWAYS use this function instead of direct LLM calls with raw data.
    """
    guardrail = DataGuardrail()
    
    # Step 1: Sanitize the data
    safe_data = guardrail.prepare_for_llm(data)
    
    # Step 2: Log what we're sending (for audit)
    print("[GUARDRAIL] Data sanitized before LLM call")
    print(f"[GUARDRAIL] Sending {len(safe_data)} safe fields to LLM")
    
    # Step 3: Make the LLM call with safe data only
    response = insight_generator.answer_question(question, safe_data)
    
    return response

# Example usage
print("SAFE LLM QUERY EXAMPLE:")
print("=" * 50)

question = "What are the key performance trends?"
data_with_sensitive_info = {
    'customer_id': 'CUST-SECRET-123',
    'on_time_rate': on_time_rate,
    'fleet_utilization': fleet_utilization,
    'satisfaction': avg_satisfaction
}

# This is the SAFE way to query the LLM
answer = safe_llm_query(insight_gen, question, data_with_sensitive_info)
print(f"\nQuestion: {question}")
print(f"Answer: {answer}")

In [ ]:
# Data Classification Helper - Identify what data is safe vs sensitive
def classify_data_fields(df: pd.DataFrame) -> Dict[str, List[str]]:
    """
    Classify DataFrame columns into safe vs sensitive categories.
    Use this to understand what data can be sent to LLMs.
    """
    guardrail = DataGuardrail()
    
    classification = {
        'SAFE_FOR_LLM': [],      # Can be sent to external LLMs
        'SENSITIVE_MASK': [],    # Must be masked before LLM
        'NEVER_SEND': []         # Never send to LLM
    }
    
    never_send_patterns = ['password', 'ssn', 'credit', 'bank', 'tax_id']
    
    for col in df.columns:
        col_lower = col.lower()
        
        # Check for highly sensitive fields
        if any(pattern in col_lower for pattern in never_send_patterns):
            classification['NEVER_SEND'].append(col)
        # Check for PII fields that need masking
        elif any(sensitive in col_lower for sensitive in guardrail.SENSITIVE_FIELDS):
            classification['SENSITIVE_MASK'].append(col)
        # Safe fields
        else:
            classification['SAFE_FOR_LLM'].append(col)
    
    return classification

# Classify customer data fields
print("DATA CLASSIFICATION FOR CUSTOMER DATA:")
print("=" * 50)
classification = classify_data_fields(customer_df)

for category, fields in classification.items():
    print(f"\n{category}:")
    if fields:
        for field in fields:
            print(f"  - {field}")
    else:
        print("  (No fields in this category - Good! No highly sensitive data found)")

### Data Guardrail Best Practices

| Do | Don't |
|---|---|
| Send aggregated KPIs (averages, totals) | Send raw customer records |
| Use masked/anonymized IDs | Send real customer IDs or names |
| Send statistical summaries | Send full DataFrames |
| Log all LLM data transfers | Skip audit trails |
| Use `safe_llm_query()` wrapper | Call LLM directly with raw data |

**Remember:** Once data is sent to an external LLM, you lose control over it. Always sanitize first!

---
## Section 9: RAG & Embeddings (Vector Search)

Implement Retrieval-Augmented Generation for context-aware AI responses.

**What is RAG?**
- **Retrieval**: Search relevant documents using vector embeddings
- **Augmented**: Add retrieved context to LLM prompts
- **Generation**: LLM generates answers using the context

**Components:**
- **Embeddings**: OpenAI `text-embedding-3-small`
- **Vector DB**: ChromaDB (local, persistent)
- **Framework**: LangChain for orchestration

In [ ]:
# Install RAG dependencies (run once)
# !pip install langchain langchain-openai langchain-community chromadb

# Import RAG Engine
from src.rag_engine import (
    LogisticsRAGEngine, 
    RAGInsightGenerator,
    DocumentProcessor
)

# Initialize RAG Engine
rag_engine = LogisticsRAGEngine(
    persist_directory="./data/vectordb",
    collection_name="penske_logistics"
)

# Check status
stats = rag_engine.get_collection_stats()
print("RAG ENGINE STATUS")
print("=" * 50)
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Step 1: Ingest Fleet Operations Data
print("INGESTING FLEET DATA")
print("=" * 50)

# Check if fleet_df exists, if not generate it
if 'fleet_df' not in dir():
    print("fleet_df not found - generating data first...")
    from src.data_generator import generate_fleet_operations
    fleet_df = generate_fleet_operations(num_records=500)
    print(f"Generated fleet_df with {len(fleet_df)} records")

result = rag_engine.ingest_fleet_data(fleet_df)

print(f"Documents added: {result['documents_added']}")
print(f"Batches processed: {result['batches_processed']}")
if result.get('errors'):
    print(f"Errors: {result['errors']}")

In [ ]:
# Step 2: Ingest Customer Data (with anonymization for privacy)
print("INGESTING CUSTOMER DATA (ANONYMIZED)")
print("=" * 50)

# Check if customer_df exists, if not generate it
if 'customer_df' not in dir():
    print("customer_df not found - generating data first...")
    from src.data_generator import generate_customer_data
    customer_df = generate_customer_data(num_customers=200)
    print(f"Generated customer_df with {len(customer_df)} records")

result = rag_engine.ingest_customer_data(customer_df, anonymize=True)

print(f"Documents added: {result['documents_added']}")
print(f"Mode: {'Anonymized' if result.get('mode') != 'mock' else 'Mock'}")

In [ ]:
# Step 3: Ingest KPI Snapshot (for historical comparison)
print("INGESTING KPI SNAPSHOT")
print("=" * 50)

# Calculate KPIs if not already defined
if 'on_time_rate' not in dir():
    on_time_rate = fleet_df['on_time_rate'].mean() * 100
if 'fleet_utilization' not in dir():
    fleet_utilization = fleet_df['load_capacity_used'].mean()
if 'avg_satisfaction' not in dir():
    avg_satisfaction = customer_df['satisfaction_score'].mean() if 'customer_df' in dir() else 7.5
if 'fuel_efficiency' not in dir():
    fuel_efficiency = fleet_df['miles_driven'].sum() / fleet_df['fuel_consumed'].sum()

current_kpis = {
    'on_time_rate': on_time_rate,
    'fleet_utilization': fleet_utilization,
    'customer_satisfaction': avg_satisfaction,
    'fuel_efficiency': fuel_efficiency,
    'total_deliveries': fleet_df['total_deliveries'].sum(),
    'total_miles': fleet_df['miles_driven'].sum()
}

result = rag_engine.ingest_kpi_snapshot(current_kpis, snapshot_name="Current_KPIs")

print(f"KPI snapshot ingested: {result['documents_added']} documents")

In [ ]:
# Step 4: Semantic Search - Find relevant documents
print("SEMANTIC SEARCH EXAMPLES")
print("=" * 50)

# Example queries
queries = [
    "Which regions have the best on-time delivery performance?",
    "What factors affect fleet utilization?",
    "Customer satisfaction trends"
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 40)
    
    results = rag_engine.search(query, k=3)
    
    for i, result in enumerate(results, 1):
        print(f"  [{i}] Score: {result['relevance_score']:.3f}")
        print(f"      Content: {result['content'][:100]}...")
        print(f"      Source: {result['metadata'].get('data_type', 'unknown')}")

In [ ]:
# Step 5: RAG-Enhanced Q&A - Combine retrieval with LLM generation
print("RAG-ENHANCED Q&A")
print("=" * 50)

# Initialize RAG Insight Generator
rag_insight = RAGInsightGenerator(rag_engine=rag_engine)

# Ask questions with context retrieval
rag_questions = [
    "What are the main performance issues in our fleet operations?",
    "How does customer satisfaction vary by industry?",
    "What regions need the most operational improvement?"
]

for question in rag_questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print("-" * 60)
    
    result = rag_insight.answer_with_context(
        question=question,
        additional_context=current_kpis,
        k=3
    )
    
    print(f"SOURCES RETRIEVED: {result['num_sources']}")
    print(f"RAG ENABLED: {result['rag_enabled']}")
    print(f"\nANSWER:\n{result['answer']}")

In [ ]:
# Step 6: Custom Document Ingestion - Add your own documents
print("CUSTOM DOCUMENT INGESTION")
print("=" * 50)

# Example: Add custom logistics knowledge
processor = DocumentProcessor(chunk_size=500, chunk_overlap=100)

custom_docs = [
    {
        "text": """
        Penske Logistics Best Practices for Fleet Management:
        1. Maintain 85%+ fleet utilization for optimal ROI
        2. Target 95%+ on-time delivery rate for customer satisfaction
        3. Regular preventive maintenance reduces breakdowns by 40%
        4. Route optimization can reduce fuel costs by 15-20%
        5. Driver training programs improve safety scores by 25%
        """,
        "metadata": {
            "source": "best_practices",
            "data_type": "knowledge_base",
            "category": "fleet_management"
        }
    },
    {
        "text": """
        Customer Retention Strategies:
        - Proactive communication on delivery status
        - Quick resolution of service issues (< 24 hours)
        - Regular business reviews with key accounts
        - Customized reporting dashboards
        - Volume-based pricing incentives
        """,
        "metadata": {
            "source": "best_practices",
            "data_type": "knowledge_base", 
            "category": "customer_retention"
        }
    }
]

result = rag_engine.add_documents(custom_docs)
print(f"Custom documents added: {result['documents_added']}")

In [ ]:
# Step 7: View Vector Store Statistics
print("VECTOR STORE STATISTICS")
print("=" * 50)

stats = rag_engine.get_collection_stats()

for key, value in stats.items():
    print(f"  {key}: {value}")

print("\n" + "=" * 50)
print("RAG IMPLEMENTATION COMPLETE!")
print("Your logistics data is now searchable via semantic embeddings.")

### RAG Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                     USER QUESTION                           │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              EMBEDDING MODEL                                │
│         (text-embedding-3-small)                            │
│         Converts question → vector                          │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              VECTOR DATABASE (ChromaDB)                     │
│         Semantic similarity search                          │
│         Returns top-k relevant documents                    │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              CONTEXT BUILDER                                │
│         Combines: Retrieved docs + Current KPIs             │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              LLM (GPT-4)                                    │
│         Generates answer using context                      │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              RESPONSE WITH SOURCES                          │
└─────────────────────────────────────────────────────────────┘
```

### Production Best Practices

| Practice | Implementation |
|----------|----------------|
| **Chunking** | 1000 chars with 200 overlap |
| **Deduplication** | Content hash-based IDs |
| **Anonymization** | Mask PII before embedding |
| **Batch Processing** | 100 docs per batch |
| **Persistence** | ChromaDB local storage |
| **Metadata Filtering** | Filter by data_type, category |

---
## Congratulations!

You've completed the Penske Logistics Analytics learning guide!

**Next steps (run these commands in your terminal):**
```bash
# Run the dashboard
streamlit run app/streamlit_dashboard.py

# Test the API
uvicorn app.api_server:app --reload
```

## Section 10: Using This Template with Real Data

This project is designed as a **reusable template**. Here's how to adapt it for your own data projects.

---
## Section 10: Using This Template with Real Data

This project is designed as a **reusable template**. Here's how to adapt it for your own data projects.

---

### Reusability Design

#### What You DON'T Need to Rewrite

| Component | Why It's Reusable |
|-----------|------------------|
| **Data Loading** | `DataLoader` accepts any CSV files |
| **RAG Engine** | Accepts any DataFrame or text documents |
| **GenAI Insights** | Works with any KPI dictionary |
| **Guardrails** | Generic PII detection patterns |
| **ML Models** | Retrain with new data, same code |

#### What You MAY Need to Adjust

| Component | When to Adjust |
|-----------|----------------|
| **Column Names** | If your data has different column names |
| **Data Generator** | Only for demo - replace with real data |
| **Visualizations** | If you want different charts |

---
### How to Load Your Own Data

There are multiple ways to integrate your real data into this template:

In [ ]:
# Option 1: Replace CSV Files (Simplest)
# Just place your CSVs in the data/ folder with same column structure
# The DataLoader will pick them up automatically

print("OPTION 1: Replace CSV Files")
print("=" * 50)
print("""
Steps:
1. Place your CSV files in the 'data/' folder
2. Name them: fleet_operations.csv, customer_data.csv, etc.
3. Ensure columns match the expected structure
4. Run the notebook - DataLoader will use your files automatically!

Expected column structures:
- fleet_operations.csv: vehicle_id, date, region, service_type, miles_driven, 
                        fuel_consumed, load_capacity_used, on_time_deliveries, total_deliveries
- customer_data.csv: customer_id, company_name, industry, region, contract_value, 
                     satisfaction_score, tenure_months
- regional_demand.csv: date, region, service_type, shipment_volume, weather_impact
""")

In [ ]:
# Option 2: Load Custom Data Directly
print("OPTION 2: Load Custom Data Directly")
print("=" * 50)

# Example: Load your own data files
# Uncomment and modify paths to use your real data

# my_fleet_data = pd.read_csv("path/to/your/fleet_data.csv")
# my_customer_data = pd.read_excel("path/to/your/customers.xlsx")

# Create datasets dictionary for use with all modules
# datasets = {
#     'fleet_operations': my_fleet_data,
#     'customer_data': my_customer_data,
#     'delivery_performance': my_delivery_data,
#     'warehouse_metrics': my_warehouse_data,
#     'regional_demand': my_demand_data
# }

# All existing code works with your data!
# analyzer = ServicePerformanceAnalyzer(datasets)
# rag_engine.ingest_fleet_data(my_fleet_data)
# predictor.train_demand_model(my_demand_data, target_col='shipment_volume')

print("""
Steps:
1. Load your data using pandas (CSV, Excel, JSON, SQL, etc.)
2. Create a datasets dictionary with your DataFrames
3. Pass to any module - they work with any data!

Example:
    my_data = pd.read_csv("my_fleet.csv")
    datasets = {'fleet_operations': my_data}
    analyzer = ServicePerformanceAnalyzer(datasets)
""")

In [ ]:
# Option 3: Different Column Names - Use Column Mapping
print("OPTION 3: Map Your Column Names")
print("=" * 50)

# If your data has different column names, simply rename them
# Example: Your data has 'delivery_time' but code expects 'on_time_rate'

# my_data = pd.read_csv("my_data.csv")
# my_data = my_data.rename(columns={
#     'delivery_time': 'on_time_rate',
#     'truck_id': 'vehicle_id',
#     'area': 'region',
#     'client_name': 'company_name',
#     'satisfaction': 'satisfaction_score'
# })

print("""
Column Mapping Example:

    my_data = pd.read_csv("my_fleet.csv")
    
    # Map your columns to expected names
    column_mapping = {
        'truck_id': 'vehicle_id',
        'delivery_date': 'date',
        'territory': 'region',
        'miles': 'miles_driven',
        'gallons': 'fuel_consumed',
        'on_time_pct': 'on_time_rate'
    }
    
    my_data = my_data.rename(columns=column_mapping)
    
    # Now use with any module!
    rag_engine.ingest_fleet_data(my_data)
""")

In [ ]:
# Option 4: Use the CustomDataLoader Module (Recommended for Production)
print("OPTION 4: Use CustomDataLoader (Production Ready)")
print("=" * 50)

# Import the custom data loader we created
from src.custom_data_loader import CustomDataLoader

# Initialize with your data path
# loader = CustomDataLoader(data_path="path/to/your/data/folder")

# Load with automatic column mapping
# my_fleet = loader.load_fleet_data(
#     file_path="my_fleet_export.csv",
#     column_mapping={
#         'truck_id': 'vehicle_id',
#         'territory': 'region'
#     }
# )

print("""
CustomDataLoader Features:
- Automatic column mapping and validation
- Support for CSV, Excel, JSON, and SQL databases
- Schema validation to ensure data quality
- Flexible date parsing
- Missing value handling

Example:
    from src.custom_data_loader import CustomDataLoader
    
    loader = CustomDataLoader(data_path="./my_data")
    
    fleet_df = loader.load_fleet_data(
        file_path="fleet_export.csv",
        column_mapping={'truck_id': 'vehicle_id'}
    )
    
    customer_df = loader.load_customer_data(
        file_path="customers.xlsx"
    )
    
    # Get all datasets ready for analytics
    datasets = loader.get_all_datasets()
""")

---
### Reusable Architecture

```
┌─────────────────────────────────────────────────────────┐
│                YOUR NEW DATA                            │
│   (CSV, Excel, Database, API)                           │
└─────────────────────┬───────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────┐
│          REUSABLE MODULES (No changes needed)           │
├─────────────────────────────────────────────────────────┤
│  • ServicePerformanceAnalyzer  - Works with any metrics │
│  • DemandForecaster            - Trains on your data    │
│  • LeadScorer / ChurnPredictor - Adapts to your columns │
│  • LogisticsRAGEngine          - Embeds any documents   │
│  • DataGuardrail               - Protects any PII       │
│  • InsightGenerator            - Analyzes any KPIs      │
└─────────────────────────────────────────────────────────┘
```

**Bottom line:** You only replace the data source. The analytics, ML, RAG, and GenAI layers are fully reusable.

In [ ]:
# PRACTICAL EXAMPLE: Complete Workflow with Your Own Data
print("COMPLETE REAL DATA WORKFLOW EXAMPLE")
print("=" * 60)

print("""
# Step-by-Step: Using This Template with Your Real Data

# ============================================================
# STEP 1: Load Your Data
# ============================================================

import pandas as pd
from src.custom_data_loader import CustomDataLoader

# Option A: Direct load
fleet_df = pd.read_csv("your_data/fleet_operations.csv")
customer_df = pd.read_excel("your_data/customers.xlsx")

# Option B: Use CustomDataLoader with column mapping
loader = CustomDataLoader(data_path="your_data/")
fleet_df = loader.load_fleet_data(
    file_path="fleet_export.csv",
    column_mapping={'truck_id': 'vehicle_id', 'territory': 'region'}
)

# ============================================================
# STEP 2: Create Datasets Dictionary
# ============================================================

datasets = {
    'fleet_operations': fleet_df,
    'customer_data': customer_df,
    'delivery_performance': delivery_df,
    'warehouse_metrics': warehouse_df,
    'regional_demand': demand_df
}

# ============================================================
# STEP 3: Run Analytics (No Code Changes!)
# ============================================================

# Performance Analysis
from src.service_performance import ServicePerformanceAnalyzer
analyzer = ServicePerformanceAnalyzer(datasets)
kpis = analyzer.calculate_fleet_kpis()

# ML Predictions
from src.resource_prediction import DemandForecaster
predictor = DemandForecaster()
results = predictor.train_demand_model(demand_df, target_col='shipment_volume')

# Customer Analytics
from src.customer_acquisition import LeadScorer, ChurnPredictor
churn = ChurnPredictor()
churn.train(customer_df)

# ============================================================
# STEP 4: RAG & GenAI (No Code Changes!)
# ============================================================

# Ingest your data to vector store
from src.rag_engine import LogisticsRAGEngine
rag = LogisticsRAGEngine()
rag.ingest_fleet_data(fleet_df)
rag.ingest_customer_data(customer_df, anonymize=True)

# Generate insights
from src.genai_insights import InsightGenerator
insight_gen = InsightGenerator()
insight = insight_gen.generate_insight(kpis)

# ============================================================
# STEP 5: Apply Guardrails (No Code Changes!)
# ============================================================

guardrail = DataGuardrail()
safe_data = guardrail.prepare_for_llm({'kpis': kpis})

print("\\nAll modules work with YOUR data - no rewrites needed!")
""")

---
### Real Data Project Checklist

Use this checklist when adapting this template for your own project:

| Step | Task | Status |
|------|------|--------|
| 1 | Prepare your data files (CSV/Excel/JSON) | ☐ |
| 2 | Identify required columns and create mapping | ☐ |
| 3 | Set up OpenAI API key for GenAI features | ☐ |
| 4 | Load data using CustomDataLoader or pandas | ☐ |
| 5 | Run performance analysis with your metrics | ☐ |
| 6 | Train ML models on your data | ☐ |
| 7 | Ingest documents to RAG vector store | ☐ |
| 8 | Configure guardrails for your PII fields | ☐ |
| 9 | Test GenAI insights with your KPIs | ☐ |
| 10 | Deploy dashboard with your data | ☐ |

---

### Best Practices for Production

| Practice | Implementation |
|----------|----------------|
| **Data Validation** | Use CustomDataLoader's schema validation |
| **PII Protection** | Always apply DataGuardrail before LLM calls |
| **API Keys** | Store in environment variables, not code |
| **Model Persistence** | Save trained models with `predictor.save_model()` |
| **Vector Store** | Use persistent ChromaDB directory |
| **Logging** | Enable logging for debugging and auditing |
| **Error Handling** | Wrap LLM calls in try/except blocks |

---

### File Structure for Your Project

```
your-project/
├── data/
│   ├── fleet_operations.csv      # Your fleet data
│   ├── customer_data.csv         # Your customer data
│   └── regional_demand.csv       # Your demand data
├── src/
│   ├── custom_data_loader.py     # Data loading (reuse as-is)
│   ├── service_performance.py    # Analytics (reuse as-is)
│   ├── resource_prediction.py    # ML models (reuse as-is)
│   ├── customer_acquisition.py   # Lead/Churn (reuse as-is)
│   ├── rag_engine.py             # RAG system (reuse as-is)
│   ├── genai_insights.py         # GenAI (reuse as-is)
│   └── data_generator.py         # DELETE (demo only)
├── notebooks/
│   └── learning_guide.ipynb      # This notebook (customize)
├── app/
│   ├── streamlit_dashboard.py    # Dashboard (customize)
│   └── api_server.py             # API (reuse as-is)
├── .env                          # API keys (create)
└── requirements.txt              # Dependencies (reuse as-is)
```

In [ ]:
# QUICK START: Minimal Code to Get Started with Your Data
print("QUICK START - Minimal Code for Your Project")
print("=" * 60)

# This is the minimal code you need to get started with your own data
# Copy this to a new notebook and customize for your project

quick_start_code = '''
# ============================================================
# QUICK START: Your Data Analytics Project
# ============================================================

import pandas as pd
import sys
import os

# Setup path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# ============================================================
# 1. LOAD YOUR DATA
# ============================================================

# Replace these with your actual data files
fleet_df = pd.read_csv("data/your_fleet_data.csv")
customer_df = pd.read_csv("data/your_customer_data.csv")

# Column mapping (if needed)
fleet_df = fleet_df.rename(columns={
    'your_vehicle_column': 'vehicle_id',
    'your_region_column': 'region'
})

# ============================================================
# 2. RUN ANALYTICS
# ============================================================

from src.service_performance import ServicePerformanceAnalyzer

datasets = {
    'fleet_operations': fleet_df,
    'customer_data': customer_df
}

analyzer = ServicePerformanceAnalyzer(datasets)

# ============================================================
# 3. TRAIN ML MODELS
# ============================================================

from src.resource_prediction import DemandForecaster

predictor = DemandForecaster()
# predictor.train_demand_model(your_demand_df, target_col='your_target')

# ============================================================
# 4. RAG SEARCH
# ============================================================

from src.rag_engine import LogisticsRAGEngine

rag = LogisticsRAGEngine(persist_directory="./data/vectordb")
rag.ingest_fleet_data(fleet_df)

# Search your data
results = rag.search("What regions have best performance?")

# ============================================================
# 5. GENAI INSIGHTS
# ============================================================

import os
os.environ['OPENAI_API_KEY'] = "your-api-key-here"

from src.genai_insights import InsightGenerator

insight_gen = InsightGenerator()
kpis = {'on_time_rate': 95.5, 'utilization': 78.3}
insight = insight_gen.generate_insight(kpis)
print(insight)
'''

print(quick_start_code)
print("\n" + "=" * 60)
print("Copy the above code to start your own project!")
print("=" * 60)

---
## Congratulations!

You've completed the **Penske Logistics Analytics Learning Guide**!

### What You've Learned:
- ✅ Data generation and exploration
- ✅ Performance analysis and KPI calculation
- ✅ ML-based demand forecasting
- ✅ Customer acquisition (lead scoring & churn prediction)
- ✅ GenAI insights with OpenAI
- ✅ Data guardrails for LLM protection
- ✅ RAG & embeddings for semantic search
- ✅ How to adapt this template for real data projects

### Next Steps (run in terminal):
```bash
# Run the interactive dashboard
streamlit run app/streamlit_dashboard.py

# Start the REST API
uvicorn app.api_server:app --reload
```

### Resources:
- **Documentation**: See README.md for full documentation
- **Deployment**: See DEPLOYMENT.md for production deployment guide
- **Tests**: Run `pytest tests/` to verify all modules work correctly

---
**This template is fully reusable. Replace the demo data with your own and start analyzing!**